# Lab 4.1 · Inner Products, Angles, and Projection

Compute lengths and angles, distinguish orthogonal from orthonormal, and derive a projection by making its residual perpendicular.

Wooseok Ha, MAS110 Lecture 4 (Fall 2026), lec04.pdf: printed slides 3–6, 12–25, 27, and 30 (PDF pages 5–8, 14–29, 31, and 34).

Maintained in [MAS110 Practice Sessions](https://github.com/statchan1106/mas110-practice-sessions). Course companion to [Foundations of LADS](https://github.com/kyunghyuncho/Foundations_of_LADS) by Wanmo Kang and Kyunghyun Cho.

Run all cells in order. NumPy handles the calculations; the final plot uses Matplotlib. Both are available in Colab.

We keep all vectors real. An inner product is symmetric and bilinear, with ⟨u,u⟩ > 0 for every nonzero u. Here the standard inner product is uᵀv. Lab 4.2 keeps the same geometry but changes the inner product to an integral. Our deterministic example v = (2,1)ᵀ and u = (1,2)ᵀ makes every calculation inspectable.

## Notation

- **⟨u,v⟩; ‖u‖; u ⊥ v**: An inner product gives one real number, ‖u‖ = √⟨u,u⟩ is its induced norm, and u ⊥ v means ⟨u,v⟩ = 0. The lecture writes |u| for this norm. In Euclidean space the default is uᵀv and the norm is ‖u‖₂; for polynomials it is the stated integral norm.

- **φ; θ; θ̂**: This chapter uses φ for an angle and θ for model coefficients in Lab 4.4. The lecture sometimes uses θ for the angle too; the distinction here prevents mixing an angle with a coefficient vector. A hat marks a minimizing coefficient, not a unit vector.

- **A; G; Q; R; P**: A stores input columns in the Euclidean QR and least-squares labs. G denotes a Gram matrix of inner products, including the polynomial metric (which the lecture locally calls A). Q stores orthonormal columns, R stores their reconstruction coefficients, and P projects onto a subspace. These matrices have different jobs and shapes.

- **u,v ∈ ℝ²; q = v/‖v‖₂**: u is the vector to project; v is a nonzero direction. q is its unit direction. Dividing by a norm requires a nonzero vector. The zero vector is orthogonal to every vector, but it has no direction or angle.

- **cos φ = uᵀv/(‖u‖₂‖v‖₂)**: The formula requires both vectors to be nonzero. Cauchy–Schwarz bounds the ratio by 1 in absolute value. Clip tiny floating-point excursions to [−1,1] before arccos; do not use clipping to hide a zero denominator.

- **p = λv; λ = uᵀv/(vᵀv); e = u − p**: λ is a scalar coefficient, p is a vector on span{v}, and e is the perpendicular residual. For unit q the same vector is p = (qᵀu)q. The coefficients λ and qᵀu differ because their basis vectors have different lengths.

- **Q ∈ ℝᵐˣᵏ; QᵀQ = Iₖ**: Columns are orthonormal, so k ≤ m. QQᵀ is the m × m projection onto Col(Q), not a meaningless matrix. It equals Iₘ only when the orthonormal columns span all of ℝᵐ; an orthogonal matrix is square.

- **W⊥; V = W ⊕ W⊥**: W⊥ contains all vectors perpendicular to every vector of W. In finite-dimensional inner-product spaces each vector has a unique decomposition into these two subspaces. Two perpendicular subspaces need not span the whole space.

- **NumPy: @; np.outer; .T**: For two 1D arrays @ is a dot product and returns a scalar. np.outer(q,q) makes the matrix qqᵀ. q.T alone still has shape (2,), so q @ q.T is a scalar, not the projection matrix.

## Key ideas

### Inner product

A scalar comparison satisfying symmetry, bilinearity, and positive definiteness.

⟨u,v⟩ = uᵀv

Keep in mind: The inner product is a scalar; elementwise products form a vector until summed.

### Norm and angle

A norm measures length; a normalized dot product compares directions.

‖u‖₂ = √(uᵀu); cos φ = ⟨u,v⟩/(‖u‖₂‖v‖₂)

Keep in mind: Zero length cannot be normalized and has no angle.

### Projection

The component on a subspace leaving a perpendicular residual.

p = (qᵀu)q; e = u − p

Keep in mind: This short formula assumes q has unit norm.

## 1. Compute the dot product

Multiply matching entries, then add. A dot product reduces two vectors to one real number.

**Predict first:** What does u * v return, and what does u @ v return?

- Line 1, `import numpy as np`: Loads NumPy. Shape: module. Operation: np supplies arrays and linear algebra.

- Line 2, `u = np.array([1., 2.])`: Stores the vector to project. Shape: u: (2,). Operation: u = (1,2)ᵀ.

- Line 3, `v = np.array([2., 1.])`: Stores the projection direction. Shape: v: (2,). Operation: v = (2,1)ᵀ, with nonzero norm.

- Line 4, `dot_uv = u @ v`: Takes the inner product. Shape: (2,) @ (2,) → scalar. Operation: 1·2 + 2·1 = 4.

In [ ]:
import numpy as np
u = np.array([1., 2.])
v = np.array([2., 1.])
dot_uv = u @ v

In [ ]:
print("u =", u, "v =", v, "dot product =", dot_uv, sep="\n")

**Read the result:** The dot product is 4. It is not a vector or an angle.

⟨u,v⟩ = 4



## 2. Turn squared lengths into lengths

The same inner product with both arguments equal gives squared length. Its square root gives length.

**Predict first:** Why is a squared norm nonnegative?

- Line 1, `norm_u = np.linalg.norm(u)`: Computes the length of u. Shape: (2,) → scalar. Operation: √(1² + 2²) = √5.

- Line 2, `norm_v = np.linalg.norm(v)`: Computes the length of v. Shape: (2,) → scalar. Operation: √(2² + 1²) = √5 > 0.

- Line 3, `q = v / norm_v`: Normalizes the nonzero direction. Shape: (2,) / scalar → (2,). Operation: q = (2/√5,1/√5)ᵀ and qᵀq = 1.

In [ ]:
norm_u = np.linalg.norm(u)
norm_v = np.linalg.norm(v)
q = v / norm_v

In [ ]:
print("norms =", norm_u, norm_v, "q =", q)

**Read the result:** q and v span the same line, but q has length 1.

‖u‖₂ = ‖v‖₂ = √5; ‖q‖₂ = 1



## 3. Compute the angle safely

Divide out both lengths. Clip rounding errors at the endpoints before converting the cosine to degrees.

**Predict first:** Would the formula work if u were zero?

- Line 1, `cos_phi = np.clip(dot_uv / (norm_u * norm_v), -1., 1.)`: Computes a direction-only comparison. Shape: scalar / scalar → scalar. Operation: cos φ = 4/5 = 0.8. Both norms here are nonzero.

- Line 2, `phi_degrees = np.degrees(np.arccos(cos_phi))`: Converts cosine to an angle. Shape: scalar radians → scalar degrees. Operation: φ ≈ 36.869898°. arccos returns radians; degrees converts units.

In [ ]:
cos_phi = np.clip(dot_uv / (norm_u * norm_v), -1., 1.)
phi_degrees = np.degrees(np.arccos(cos_phi))

In [ ]:
print("cosine =", cos_phi, "angle in degrees =", phi_degrees)

**Read the result:** A positive dot product gives an acute angle between nonzero vectors.

cos φ = 4/5; φ ≈ 36.87°



## 4. Project onto the line

The scalar qᵀu is the coordinate along the unit direction. Multiplying by q returns to ambient vector space.

**Predict first:** Which value is a coordinate, and which is a vector?

- Line 1, `coordinate = q @ u`: Finds the coordinate along q. Shape: (2,) @ (2,) → scalar. Operation: qᵀu = 4/√5.

- Line 2, `projected = coordinate * q`: Forms the projected vector. Shape: scalar × (2,) → (2,). Operation: p = (8/5,4/5)ᵀ.

- Line 3, `residual = u - projected`: Keeps the part not explained by the line. Shape: (2,) − (2,) → (2,). Operation: e = (−3/5,6/5)ᵀ.

- Line 4, `perpendicular = q @ residual`: Checks perpendicularity. Shape: (2,) @ (2,) → scalar. Operation: qᵀe ≈ 0 within floating-point tolerance.

In [ ]:
coordinate = q @ u
projected = coordinate * q
residual = u - projected
perpendicular = q @ residual

In [ ]:
print("coordinate =", coordinate, "projection =", projected, "residual =", residual, "q.T @ residual =", perpendicular, sep="\n")

**Read the result:** u = p + e, with p on span{v} and e perpendicular to v.

u = p + e; vᵀe = 0



## 5. Make the projection a matrix

An outer product builds a map acting on any input vector. The short basis has one column, so its projection is not the full identity.

**Predict first:** Why is P 2 × 2 while Q is 2 × 1?

- Line 1, `Q = q[:, None]`: Makes one explicit column. Shape: (2,) → Q: (2, 1). Operation: [:, None] adds a column dimension; .T on q alone would not.

- Line 2, `P = Q @ Q.T`: Forms the ambient projection. Shape: (2, 1) @ (1, 2) → (2, 2). Operation: P = qqᵀ = [[4,2],[2,1]]/5.

- Line 3, `projected_again = P @ u`: Applies the map to u. Shape: (2, 2) @ (2,) → (2,). Operation: P @ u gives (8/5,4/5)ᵀ again.

- Line 4, `projector_check = np.allclose(P.T, P) and np.allclose(P @ P, P)`: Checks two defining matrix properties. Shape: two comparisons → boolean. Operation: Pᵀ ≈ P and P² ≈ P; projector_check = True.

In [ ]:
Q = q[:, None]
P = Q @ Q.T
projected_again = P @ u
projector_check = np.allclose(P.T, P) and np.allclose(P @ P, P)

In [ ]:
print("Q =", Q, "P =", P, "P @ u =", projected_again, "projector check =", projector_check, sep="\n")

**Read the result:** QᵀQ = I₁ but QQᵀ = P ≠ I₂. The range is a line, not all of ℝ².

QᵀQ = [1]; QQᵀ = P



## 6. Verify the orthogonal decomposition

The squared lengths add because the components are perpendicular. This is the reason the projection is closest.

**Predict first:** Would lengths themselves add in the same way?

- Line 1, `squared_parts = np.array([projected @ projected, residual @ residual])`: Computes two squared lengths. Shape: two scalars → (2,). Operation: ‖p‖₂² = 16/5 and ‖e‖₂² = 9/5.

- Line 2, `pythagoras = np.allclose(u @ u, squared_parts.sum())`: Checks Pythagoras. Shape: scalar comparison → boolean. Operation: 16/5 + 9/5 = 5 = ‖u‖₂².

- Line 3, `q_perp = np.array([-q[1], q[0]])`: Chooses a unit perpendicular direction. Shape: (2,). Operation: Rotate q by 90°: (−1/√5,2/√5)ᵀ.

- Line 4, `Q_full = np.column_stack((q, q_perp))`: Completes an orthonormal basis. Shape: two (2,) columns → (2, 2). Operation: Q_full is square; both Q_fullᵀQ_full and Q_fullQ_fullᵀ equal I₂.

In [ ]:
squared_parts = np.array([projected @ projected, residual @ residual])
pythagoras = np.allclose(u @ u, squared_parts.sum())
q_perp = np.array([-q[1], q[0]])
Q_full = np.column_stack((q, q_perp))

In [ ]:
print("squared parts =", squared_parts, "Pythagoras =", pythagoras, "Q_full =", Q_full, sep="\n")

**Read the result:** Adding the perpendicular direction fills ℝ². The earlier line projection remains useful as one part of this decomposition.

‖u‖₂² = 16/5 + 9/5 = 5; Q_fullQ_fullᵀ = I₂



## Why the dot product determines the geometry

### Cauchy–Schwarz makes the angle formula possible

If v ≠ 0, 0 ≤ ‖u − tv‖₂² = ‖u‖₂² − 2t⟨u,v⟩ + t²‖v‖₂² for every real t. At t = ⟨u,v⟩/‖v‖₂², this implies ⟨u,v⟩² ≤ ‖u‖₂²‖v‖₂². The v = 0 case gives 0 ≤ 0 directly.

Thus the cosine ratio lies in [−1,1] for nonzero vectors. Scaling a vector by a positive number changes its length, but not its angle with another vector. A negative scale reverses its direction.

|⟨u,v⟩| ≤ ‖u‖₂‖v‖₂; here cos φ = 4/5

### A unit direction changes the coordinate, not the line

The vectors v and q = v/‖v‖₂ span the same line. A coordinate along v is λ = 4/5, while the coordinate along q is qᵀu = 4/√5. Multiplying either coordinate by its associated basis vector gives p = (8/5,4/5)ᵀ.

Orthogonal only means a zero inner product. Orthonormal additionally means unit length. Projection by an inner product alone uses a unit direction; otherwise the denominator ⟨v,v⟩ must remain.

p = (uᵀv)/(vᵀv) · v = (qᵀu)q

### The perpendicular residual certifies the nearest point

With e = u − λv and λ = ⟨u,v⟩/⟨v,v⟩, ⟨e,v⟩ = 0. Any other point μv differs from p only along the line, so its error is the sum of two perpendicular components.

Here e = (−3/5,6/5)ᵀ, with squared length 9/5. Every μ other than 4/5 adds a positive squared distance along the line.

‖u − μv‖₂² = ‖e‖₂² + (μ − λ)²‖v‖₂²

### QQᵀ is a projection; completing Q changes its range

Because QᵀQ = Iₖ, P = QQᵀ satisfies Pᵀ = P and P² = Q(QᵀQ)Qᵀ = P. Also Pz lies in Col(Q) and Qᵀ(z − Pz) = 0. That is exactly orthogonal projection.

The original 4.1 notebook calls a rectangular Q’s QQᵀ meaningless. The corrected interpretation is a projection onto the chosen column space. Completing Q to a square orthogonal basis makes its range the whole space and QQᵀ = Iₘ.

q = (2,1)ᵀ/√5 ⇒ P = 1/5 · [[4,2],[2,1]] ≠ I₂

### Perpendicular subspaces and complements are different claims

For U = span{(1,0,0)ᵀ} and W = span{(0,1,0)ᵀ}, U ⊥ W. But U⊥ is the entire yz-plane; W is only one line in it. A complement must also account for the remaining direction.

For any finite-dimensional subspace W, projection gives u = Pu + (I − P)u. The intersection W ∩ W⊥ is {0}, since a vector perpendicular to itself has zero norm. This proves uniqueness of the decomposition.



## Numerical checks

These checks verify the example and the correction described above. They support the algebraic arguments and do not replace them.

In [ ]:
assert np.allclose(u * v, [2, 2]) and np.isclose(dot_uv, 4)
assert np.allclose([norm_u, norm_v], np.sqrt(5))
assert np.isclose(cos_phi, 4 / 5)
assert np.isclose(phi_degrees, 36.86989764584402)
assert np.allclose(projected, [8 / 5, 4 / 5])
assert np.allclose(residual, [-3 / 5, 6 / 5])
assert np.isclose(v @ residual, 0, atol=1e-12)
assert Q.shape == (2, 1) and P.shape == (2, 2)
assert np.allclose(P, np.array([[4, 2], [2, 1]]) / 5)
assert projector_check and pythagoras
assert np.allclose(squared_parts, [16 / 5, 9 / 5])
assert np.allclose(Q_full.T @ Q_full, np.eye(2))
assert np.allclose(Q_full @ Q_full.T, np.eye(2))
def unit_direction(vector):
    vector = np.asarray(vector, dtype=float)
    norm = np.linalg.norm(vector)
    if norm == 0:
        raise ValueError("A zero vector has no unit direction.")
    return vector / norm
try:
    unit_direction(np.zeros(2))
except ValueError:
    pass
else:
    raise AssertionError("Expected a zero-vector guard.")
for candidate in [[0, 0], [2, -1], [3, 1], [-1, -2]]:
    candidate = np.array(candidate, dtype=float)
    p = P @ candidate
    e = candidate - p
    assert np.isclose(v @ e, 0, atol=1e-12)
    assert np.isclose(candidate @ candidate, p @ p + e @ e)
print("All inner-product checks passed.")

## Try a different direction and see the vectors

Change v_new and u_new. A new line changes the projection matrix; positive scaling of v_new preserves the line and its projection. The zero direction is rejected before division.

In [ ]:
v_new = np.array([1., -1.])
u_new = np.array([2., 1.])
q_new = unit_direction(v_new)
p_new = (q_new @ u_new) * q_new
e_new = u_new - p_new
assert np.isclose(v_new @ e_new, 0, atol=1e-12)
assert np.allclose(np.outer(q_new, q_new), np.outer(unit_direction(3 * v_new), unit_direction(3 * v_new)))
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
for vector, label in [(u_new, "u"), (p_new, "p"), (v_new, "v")]:
    ax.quiver(0, 0, *vector, angles="xy", scale_units="xy", scale=1, label=label)
ax.plot([p_new[0], u_new[0]], [p_new[1], u_new[1]], "--", label="residual e")
ax.set(xlim=(-2, 3), ylim=(-2, 3), xlabel="first coordinate", ylabel="second coordinate")
ax.set_aspect("equal")
ax.legend()
plt.show()

## Check your understanding

**Is the zero vector perpendicular to v? Does it have an angle with v?**

<details><summary>Show explanation</summary>

Its dot product with v is zero, so it is orthogonal. The angle is undefined because its norm is zero. The explorer reports this case explicitly.

</details>

**Can you write the projection as (uᵀv)v?**

<details><summary>Show explanation</summary>

Only if v is a unit vector. Otherwise divide by vᵀv. For our v, this denominator is 5.

</details>

**Are λ and p interchangeable?**

<details><summary>Show explanation</summary>

No. λ = 4/5 is one coordinate; p = λv = (8/5,4/5)ᵀ is the projected vector. Changing the basis vector’s length changes the coordinate but not p.

</details>

**Why is QQᵀ useful when Q has fewer columns than rows?**

<details><summary>Show explanation</summary>

It projects onto Col(Q). QᵀQ checks orthonormality in coordinate space; QQᵀ acts in the ambient space. They have different shapes and purposes.

</details>

## Code references

- [NumPy: outer products](https://numpy.org/doc/stable/reference/generated/numpy.outer.html)